In [23]:
try:
    print(f"Spark version {sc.version}")
except:
    %run ../spark-instance.ipynb

SparkConf created
Started SparkSession
Spark version 3.5.3
You should be able to access the Spark UI at: https://dacs-compute-gate.ewi.utwente.nl:9999/user/g.luvizottocesar@utwente.nl/proxy/4040/stages/
Note that you may have to Enable extensions first via the Extension Manager.


In [22]:
clean_spark()

CLEANING SPARK INSTANCE...


In [24]:
from datetime import datetime

# installed pkg imports
import numpy as np
import pandas as pd

import pyspark.sql.types as pst
from pyspark.sql import functions as psf
from pyspark.sql.window import Window

from census_helper import download_date

## QUIC input

In [25]:
ts = datetime(2025, 6, 3)  # this is the latest we have now...
# the dataset of this date, in parquet format, has:
# Total Size: 2.2 TiB
# Total Objects: 1025

# virtual hosts data set uses names in the scan.
# more info: https://docs.censys.com/docs/ls-intro-virtual-hosts

# how they obtain the names (from their latest paper): https://doi.org/10.1145/3718958.3754344
# "Censys performs HTTP(S) scans against known names, which it collects from public CT logs, HTTP redirects, and third-party passive DNS subscriptions.
# We initially began to perform name-based scanning as an extension to IP-based scanning by creating a new asset type—Virtual Host"
CENSYS_BASE_PATH_FMT = "s3a://censys/dataset=universal-internet-dataset-v2-ipv4-virtual-hosts/format=parquet/"
CENSYS_PATH_FMT = os.path.join(CENSYS_BASE_PATH_FMT, "year={year}/month={month:02d}/day={day:02d}")

In [26]:
censys_path = CENSYS_PATH_FMT.format(year=ts.year, month=ts.month, day=ts.day)
censys_vh_df = spark.read.parquet(censys_path)

In [32]:
censys_vh_df.select("host_identifier.ipv4", "host_identifier.name").show(truncate=False)

+---------------+----------------------------------------------------+
|ipv4           |name                                                |
+---------------+----------------------------------------------------+
|147.81.11.183  |dhcp-147-81-11-183.hawaiiantel.net                  |
|103.4.75.81    |test5.demo.idt.pf                                   |
|184.28.179.245 |a184-28-179-245.deploy.static.akamaitechnologies.com|
|206.87.224.50  |www.sandbox.neuroscience.ubc.ca                     |
|209.54.113.150 |www.getflushed.ca                                   |
|209.54.113.159 |www.lucidcapital.slant.is                           |
|23.180.104.243 |mail.dispatchbvsltd.ca                              |
|64.34.50.39    |dlccontinentalmortgage.ca                           |
|67.231.16.157  |cpcalendars.brightoncollege.ca                      |
|67.231.16.180  |webmail.akirawebhosting.com                         |
|67.231.28.131  |mail.gnumatrix.com                                  |
|71.19

In [33]:
# checking if there are more names attached to the same IP address
censys_vh_df.groupBy("host_identifier.ipv4", "host_identifier.name").count().sort("count", ascending=False).show(truncate=False)

+--------------+-------------------------------------------------------------------------+-----+
|ipv4          |name                                                                     |count|
+--------------+-------------------------------------------------------------------------+-----+
|172.65.90.27  |18c32dd7b2d963c07a6cc38586637ee9.fedramp.r2.cloudflarestorage.com        |1    |
|172.64.150.67 |anelsonhaframeworksebstocraftgasme.tebex.io                              |1    |
|172.66.47.110 |legend-ek5.pages.dev                                                     |1    |
|172.66.0.70   |jukeboxomnibus.one                                                       |1    |
|34.117.166.186|sampler-92-wave-1-0-12525034100419702288.http-01.preprod.haplorrhini.com |1    |
|172.67.135.224|portainer.huydevs.com                                                    |1    |
|172.66.47.16  |customerwebservice.pages.dev                                             |1    |
|172.67.153.210|collinyzbrc.bl

In [30]:
# in case you want to see what's more into the censys virtual hosts dataset:
censys_vh_df.printSchema()

root
 |-- host_identifier: struct (nullable = true)
 |    |-- ipv4: string (nullable = true)
 |    |-- ipv6: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- services: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- port: long (nullable = true)
 |    |    |-- transport: string (nullable = true)
 |    |    |-- service_name: string (nullable = true)
 |    |    |-- perspective: string (nullable = true)
 |    |    |-- observed_at: timestamp (nullable = true)
 |    |    |-- tls: struct (nullable = true)
 |    |    |    |-- version_selected: string (nullable = true)
 |    |    |    |-- cipher_selected: string (nullable = true)
 |    |    |    |-- certificates: struct (nullable = true)
 |    |    |    |    |-- leaf_fp_sha_256: binary (nullable = true)
 |    |    |    |    |-- chain_fps_sha_256: array (nullable = true)
 |    |    |    |    |    |-- element: binary (containsNull = true)
 |    |    |    |    |-- leaf_data: struct (null

## ZMap ports

In [3]:
ts = datetime(2025, 10, 7)

CENSYS_BASE_PATH_FMT = "s3a://censys/dataset=universal-internet-dataset-v2-ipv4/format=parquet/"
CENSYS_PATH_FMT = os.path.join(CENSYS_BASE_PATH_FMT, "year={year}/month={month:02d}/day={day:02d}")

In [4]:
censys_path = CENSYS_PATH_FMT.format(year=ts.year, month=ts.month, day=ts.day)
censys_df = spark.read.parquet(censys_path)

In [7]:
# define LACeS struct

# locations of anycast sites
location_struct = pst.StructType([
    pst.StructField("city", pst.StringType(), True),
    pst.StructField("code_country", pst.StringType(), True), 
    pst.StructField("id", pst.StringType(), True),
    pst.StructField("latitude", pst.DoubleType(), True), 
    pst.StructField("longitude", pst.DoubleType(), True)
])

# schema
full_schema = pst.StructType([
    pst.StructField("prefix", pst.StringType(), True),
    pst.StructField("AB_ICMPv4", pst.LongType(), True),
    pst.StructField("AB_TCPv4", pst.LongType(), True),
    pst.StructField("AB_DNSv4", pst.LongType(), True),
    pst.StructField("GCD_ICMPv4", pst.LongType(), True),
    pst.StructField("GCD_TCPv4", pst.LongType(), True),
    pst.StructField("partial", pst.BooleanType(), True),
    pst.StructField("backing_prefix", pst.StringType(), True),
    pst.StructField("ASN", pst.StringType(), True),
    # list of location structs
    pst.StructField("locations", pst.ArrayType(location_struct), True) 
])

In [9]:
anycast_pdf = download_date(ts.year, ts.month, ts.day, "v4")
anycast_pdf = anycast_pdf[anycast_pdf['GCD_ICMPv4'] > 1] # filter on GCD-confirmed (high accuracy)

anycast_pdf.head()

,prefix,AB_ICMPv4,AB_TCPv4,AB_DNSv4,GCD_ICMPv4,GCD_TCPv4,partial,backing_prefix,ASN,locations
0,1.0.0.0/24,29,28,26,65,26,False,1.0.0.0/24,13335,"[{'city': 'Johannesburg', 'country_code': 'ZA'..."
1,1.1.1.0/24,29,28,28,66,28,False,1.1.1.0/24,13335,"[{'city': 'Bangalore', 'country_code': 'IN', '..."
4,1.12.0.0/24,5,0,4,5,0,False,1.12.0.0/20,132203_45090,"[{'city': 'Hong Kong', 'country_code': 'HK', '..."
5,1.12.12.0/24,3,1,0,3,1,False,1.12.0.0/20,132203_45090,"[{'city': 'Hong Kong', 'country_code': 'HK', '..."
6,1.12.13.0/24,3,0,2,3,0,False,1.12.0.0/20,132203_45090,"[{'city': 'Hong Kong', 'country_code': 'HK', '..."


In [11]:
anycast_spark_df = spark.createDataFrame(anycast_pdf, schema=full_schema)

anycast_spark_df.printSchema()
anycast_spark_df.show(1)

root
 |-- prefix: string (nullable = true)
 |-- AB_ICMPv4: long (nullable = true)
 |-- AB_TCPv4: long (nullable = true)
 |-- AB_DNSv4: long (nullable = true)
 |-- GCD_ICMPv4: long (nullable = true)
 |-- GCD_TCPv4: long (nullable = true)
 |-- partial: boolean (nullable = true)
 |-- backing_prefix: string (nullable = true)
 |-- ASN: string (nullable = true)
 |-- locations: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- city: string (nullable = true)
 |    |    |-- code_country: string (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- latitude: double (nullable = true)
 |    |    |-- longitude: double (nullable = true)

+----------+---------+--------+--------+----------+---------+-------+--------------+-----+--------------------+
|    prefix|AB_ICMPv4|AB_TCPv4|AB_DNSv4|GCD_ICMPv4|GCD_TCPv4|partial|backing_prefix|  ASN|           locations|
+----------+---------+--------+--------+----------+---------+-------+--------------+

In [14]:
censys_df = censys_df.withColumn(
    "prefix",
    psf.concat(
        psf.substring_index("host_identifier.ipv4", ".", 3), 
        psf.lit(".0/24")
    )
)

censys_selectives_df = censys_df.select(
    "prefix",
    "host_identifier",
    "services.port",
    "services.transport",
    "services.service_name",
    "autonomous_system", # contains more than just ASN
)


anycast_selectives_df = anycast_spark_df.select(
    "prefix", 
    "GCD_ICMPv4", 
    "partial", 
    #"backing_prefix", 
    #"ASN"
)

censys_anycast_df = censys_selectives_df.join(
    psf.broadcast(anycast_selectives_df),
    on="prefix",
    how="inner"
).select(
    "prefix",
    "GCD_ICMPv4",
    "partial",
    "host_identifier",
    "port",
    "transport",
    "service_name",
    "autonomous_system",
)

try:
    censys_anycast_df.unpersist()
except:
    pass

censys_anycast_df.persist()

DataFrame[prefix: string, GCD_ICMPv4: bigint, partial: boolean, host_identifier: struct<ipv4:string,ipv6:string,name:string>, port: array<bigint>, transport: array<string>, service_name: array<string>, autonomous_system: struct<asn:bigint,description:string,bgp_prefix:string,name:string,country_code:string,organization:string>]

In [15]:
censys_anycast_df.select("port").show()

+--------------------+
|                port|
+--------------------+
|[80, 8082, 8083, ...|
|           [80, 443]|
|           [80, 443]|
|                [80]|
|                [80]|
|[80, 443, 2052, 2...|
|[80, 443, 2052, 2...|
|[80, 443, 2052, 2...|
|[80, 443, 2052, 2...|
|[80, 443, 2052, 2...|
|[80, 443, 2052, 2...|
|[80, 443, 2052, 2...|
|[80, 443, 2052, 2...|
|[80, 443, 2052, 2...|
|[80, 443, 2052, 2...|
|[80, 443, 2052, 2...|
|[80, 443, 2052, 2...|
|[80, 443, 2052, 2...|
|[80, 443, 2052, 2...|
|[22, 80, 443, 143...|
+--------------------+
only showing top 20 rows



In [19]:
grouped_df = censys_anycast_df.select(psf.explode_outer("port").alias("port")).groupBy("port").count()

total = grouped_df.agg(psf.sum("count")).collect()[0][0]

w = Window.orderBy(psf.col("count").desc())

grouped_df.sort("count", ascending=False
).withColumn(
    "percent", (psf.col("count") / psf.lit(total)) * 100
).withColumn(
    "cumulative_percent", psf.sum("percent").over(w)
).show(50)

+-----+-------+-------------------+------------------+
| port|  count|            percent|cumulative_percent|
+-----+-------+-------------------+------------------+
|  443|1797815| 10.996464804908573|10.996464804908573|
|   80|1577549|   9.64919196720392|20.645656772112496|
| 8080| 748122|  4.575942042300132|25.221598814412626|
| 8443| 745009|  4.556901153811784| 29.77849996822441|
| 2096| 737197| 4.5091184937183115| 34.28761846194272|
| 8880| 733068| 4.4838631681261525| 38.77148163006888|
| 2083| 733015|  4.483538989812666| 43.25502061988154|
| 2087| 732894|  4.482798884606405|47.737819504487945|
| 2082| 732863|  4.482609270875875| 52.22042877536382|
| 2086| 732860|  4.482590921160018| 56.70301969652384|
| 2095| 732836|  4.482444123433156| 61.18546381995699|
| 2053| 732510|  4.480450120976618|  65.6659139409336|
| 2052| 727856|   4.45198359510936| 70.11789753604296|
| 9801| 385593|  2.358507328907096| 72.47640486495006|
| 1445| 385500| 2.3579384877155074| 74.83434335266556|
| 6599| 38

In [ ]:
# https://developers.cloudflare.com/fundamentals/reference/network-ports/
cloudflare_ports = ["80","8080","8880","2052","2082","2086","2095","443","2053","2083","2087","2096","8443","2052","2053","2082","2083","2086","2087","2095","2096","8880","8443"]

In [21]:
# eliminating overburden of HTTP and using additional databases ports
ports_to_scan = set([
    "443", "80",
    "9801", "1445", "6599", 
    "53", "25", "110", "993", "587", "143", "995", "465",
    "21", "22", 
    "3306", "5432", "8432", "6379", "27017", "9042", "1433",
    "389", "636", "3268", "3269"
])
print(list(ports_to_scan))

['21', '53', '80', '3306', '143', '5432', '6379', '1445', '27017', '389', '9801', '587', '25', '443', '9042', '3269', '22', '110', '6599', '636', '3268', '465', '993', '1433', '8432', '995']
